In [0]:
import time
import uuid
from datetime import datetime
from zoneinfo import ZoneInfo
from pyspark.sql import functions as F

SOURCE_CATALOG  = "fr_neon"
SOURCE_SCHEMA   = "public"
SOURCE_DATABASE = "unifor"           
BASE_VOLUME     = "/Volumes/meu_catalog/landing/postgres"

ADD_AUDIT_COLS  = True               
TZ              = ZoneInfo("America/Sao_Paulo")

CONTROL_TABLE   = "meu_catalog.landing.ingestion_manifest"  # controle Delta centralizado

RUN_ID   = str(uuid.uuid4())
NOW      = datetime.now(TZ)
DATE_DIR = f"{NOW:%Y}/{NOW:%Y-%m}/{NOW:%Y-%m-%d}"
TS       = f"{NOW:%Y_%m_%d_%H_%M_%S}"

CONTROL_SCHEMA = (
    "run_id string, ingested_at string, source_catalog string, source_schema string, "
    "source_database string, source_table string, status string, row_count long, "
    "file_name string, file_path string, file_size_bytes long, duration_sec double, error string"
)

def write_single_parquet(df, target_dir: str, filename: str) -> str:
    """Escreve o DataFrame como UM arquivo .parquet nomeado dentro de target_dir."""
    tmp_dir = f"{target_dir}/_tmp_{filename}"
    dbutils.fs.mkdirs(target_dir)
    df.coalesce(1).write.mode("overwrite").parquet(tmp_dir)
    part = [f for f in dbutils.fs.ls(tmp_dir) if f.name.endswith(".parquet")][0]
    final_path = f"{target_dir}/{filename}"
    dbutils.fs.mv(part.path, final_path)
    dbutils.fs.rm(tmp_dir, recurse=True)
    return final_path

def file_size(path: str):
    try:
        return dbutils.fs.ls(path)[0].size
    except Exception:
        return None

objs = (
    spark.table(f"{SOURCE_CATALOG}.information_schema.tables")
    .where(F.col("table_schema") == SOURCE_SCHEMA)
    .select("table_name", "table_type")
)

tables = [r["table_name"] for r in objs.orderBy("table_name").collect()]
print(f"run_id={RUN_ID}")
print(f"{len(tables)} objeto(s) para ingerir: {tables}")

control_rows = []

for tbl in tables:
    full_name = f"{SOURCE_CATALOG}.{SOURCE_SCHEMA}.{tbl}"
    table_dir = f"{BASE_VOLUME}/{SOURCE_DATABASE}/{tbl}/{DATE_DIR}"
    data_file = f"postgres_{TS}-{tbl}.parquet"
    data_path = f"{table_dir}/{data_file}"
    t0 = time.time()
    try:
        df = spark.read.table(full_name)
        if ADD_AUDIT_COLS:
            df = (
                df.withColumn("_run_id", F.lit(RUN_ID))
                  .withColumn("_source_table", F.lit(full_name))
                  .withColumn("_ingested_at", F.lit(NOW.isoformat()))
            )
        n = df.count()
        data_path = write_single_parquet(df, table_dir, data_file)
        dur  = round(time.time() - t0, 2)
        size = file_size(data_path)
        print(f"[OK]   {tbl:<20} {n:>8} linhas  {dur:>6}s  -> {data_path}")
        control_rows.append((RUN_ID, NOW.isoformat(), SOURCE_CATALOG, SOURCE_SCHEMA,
                             SOURCE_DATABASE, tbl, "OK", n, data_file, data_path, size, dur, ""))
    except Exception as e:
        dur = round(time.time() - t0, 2)
        print(f"[ERRO] {tbl:<20} -> {str(e)[:200]}")
        control_rows.append((RUN_ID, NOW.isoformat(), SOURCE_CATALOG, SOURCE_SCHEMA,
                             SOURCE_DATABASE, tbl, "ERRO", None, data_file, data_path, None, dur, str(e)[:500]))


control = spark.createDataFrame(control_rows, schema=CONTROL_SCHEMA)

print("\n===== RESUMO DO LOTE =====")
control.orderBy("status", "source_table").show(truncate=False)

(
    control.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(CONTROL_TABLE)
)
print(f"Registrado em {CONTROL_TABLE} (append).")



In [0]:
%sql
create schema meu_catalog.bronze


In [0]:

ok    = control.where("status = 'OK'").count()
falha = control.where("status = 'ERRO'").count()
total = control.where("status = 'OK'").agg(F.sum("row_count")).first()[0] or 0
print(f"concluído: {ok} OK / {falha} com erro | {total} linhas | run_id={RUN_ID}")

#   SELECT * FROM meu_catalog.landing.ingestion_manifest
#   ORDER BY ingested_at DESC LIMIT 50;


#   SELECT ingested_at, source_table, error
#   FROM meu_catalog.landing.ingestion_manifest
#   WHERE status = 'ERRO' ORDER BY ingested_at DESC;

#   SELECT to_date(ingested_at) dia, count(*) arquivos,
#          sum(row_count) linhas, round(sum(file_size_bytes)/1024/1024,2) mb
#   FROM meu_catalog.landing.ingestion_manifest
#   WHERE status = 'OK' GROUP BY 1 ORDER BY 1 DESC;